这个脚本的目的主要是为了从psm中得到经典和非经典的小肽对应的psm，并分别将其分为特异性肽段以及非特异性肽段

In [1]:
source("/rd1/user/lit/project/sORFs/sORFs.utils.R")
source("~/bin/lit_utils.R")
lib_text()
lib_plot()
output_path <- "../results/S3"
create_path(output_path)

In [2]:
# 函数
unfold <- function(df){
    # 分组并且展开
    df$Group_id <- 1:nrow(df)
    df %>% separate_rows(`Mapped.Proteins`, convert = TRUE,sep=", ") -> tmp
    tmp %>% mutate(Protein=`Mapped.Proteins`,
                   `Protein.ID`=`Mapped.Proteins`,
                   Gene=`Mapped.Proteins`,
                   `Entry.Name`=`Mapped.Proteins`,
                   `Protein.Description`=`Mapped.Proteins`) -> tmp_1
    rbind(df,tmp_1) -> df_unfold
    return(df_unfold)
}

In [3]:
total_psm_path="../results/S2/total_psm.txt"
noncano_sep_psm_path="../results/S2/all_sample_sep_unique_psm.txt"
psm_contain_cano_sep_path="../results/S3/psm_contain_cano_sep.txt"
uniprot.human.sep.id.path="../results/S3/uniprot.human.sep.id.txt"
one_sample_psm_path="/rd1/user/lit/project/sORFs/analysis/20250523_human_ms_run/output/db_search_20250326/Trypsin/CAD20241224licq_BSEP_DDA_60min_21pcw_1_PAGE_T_T_Slot1_46_1_4747_d/psm.tsv"

# 经典小肽

In [4]:
fread_c(psm_contain_cano_sep_path) -> psm_contain_cano_sep
read.table(uniprot.human.sep.id.path) -> uniprot.human.sep.id
read_psm(one_sample_psm_path) -> psm
colnames(psm)
psm_contain_cano_sep -> df
colnames(df) <- colnames(psm)

[1] "Spectrum"                    "Spectrum.File"              
 [3] "Peptide"                     "Modified.Peptide"           
 [5] "Extended.Peptide"            "Prev.AA"                    
 [7] "Next.AA"                     "Peptide.Length"             
 [9] "Charge"                      "Retention"                  
[11] "Observed.Mass"               "Calibrated.Observed.Mass"   
[13] "Observed.M.Z"                "Calibrated.Observed.M.Z"    
[15] "Calculated.Peptide.Mass"     "Calculated.M.Z"             
[17] "Delta.Mass"                  "SpectralSim"                
[19] "RTScore"                     "Expectation"                
[21] "Hyperscore"                  "Nextscore"                  
[23] "Probability"                 "Number.of.Enzymatic.Termini"
[25] "Number.of.Missed.Cleavages"  "Protein.Start"              
[27] "Protein.End"                 "Intensity"                  
[29] "Assigned.Modifications"      "Observed.Modifications"     
[31] "Class"                       "Ion.Mobility"               
[33] "Purity"                      "Is.Unique"                  
[35] "Protein"                     "Protein.ID"                 
[37] "Entry.Name"                  "Gene"                       
[39] "Protein.Description"         "Mapped.Genes"               
[41] "Mapped.Proteins"             "Sample"

In [5]:
# 分组并且展开
unfold(df)->df_unfold
# 先把含有长度大于150aa的注释蛋白组别找出来
df_unfold %>% filter(grepl("^sp",Protein)) %>% filter(!Protein %in% uniprot.human.sep.id$V1) -> df_anno_pro
unique(df_anno_pro$Group_id) -> df_anno_pro_gi
# 特异性+非特异性总共有多少经典的SEP
df_unfold %>% filter(Protein %in% uniprot.human.sep.id$V1) -> df_unfold_target
n_distinct(df_unfold_target$Protein)
# 特异性+非特异性分别是哪些
df_unfold_target %>% group_by(Group_id) %>% filter(n()==1 & !Group_id %in% df_anno_pro_gi) -> df_unique
unique(df_unique$Group_id) -> df_unique_gi
df_unfold_target %>% group_by(Group_id) %>% filter(n()>1 | Group_id %in% df_anno_pro_gi) -> df_multiple
unique(df_multiple$Group_id) -> df_multiple_gi
# 合并结果
df_unfold_target %>% mutate(PSM_type=case_when(
    Group_id %in% df_unique_gi ~ "Unique",
    Group_id %in% df_multiple_gi ~ "NonUnique",
)) -> df_unfold_target_add_PSM_type
# 数量check
df_unfold_target_add_PSM_type %>% group_by(PSM_type) %>% summarise(n_distinct=n_distinct(Protein))
# 导出
fwrite(df_unfold_target_add_PSM_type,o("psm_cano_sep_all.txt"),sep='\t')

[1] 2023

PSM_type,n_distinct
<chr>,<int>
NonUnique,1479
Unique,838


In [6]:
head(df_unfold_target_add_PSM_type)

Spectrum                                                                       
1 CAD20250514licq_BSEP_DDA_60min_21pcw_3_PCP_T_AC_Slot2-42_1_7070.102787.102787.3
2 CAD20250514licq_BSEP_DDA_60min_21pcw_3_PCP_T_AC_Slot2-42_1_7070.102885.102885.2
3 CAD20250514licq_BSEP_DDA_60min_21pcw_3_PCP_T_AC_Slot2-42_1_7070.102961.102961.2
4 CAD20250514licq_BSEP_DDA_60min_21pcw_3_PCP_T_AC_Slot2-42_1_7070.103262.103262.3
5 CAD20250514licq_BSEP_DDA_60min_21pcw_3_PCP_T_AC_Slot2-42_1_7070.103416.103416.3
6 CAD20250514licq_BSEP_DDA_60min_21pcw_3_PCP_T_AC_Slot2-42_1_7070.10382.10382.1  
  Spectrum.File                                                                   
1 interact-CAD20250514licq_BSEP_DDA_60min_21pcw_3_PCP_T_AC_Slot2-42_1_7070.pep.xml
2 interact-CAD20250514licq_BSEP_DDA_60min_21pcw_3_PCP_T_AC_Slot2-42_1_7070.pep.xml
3 interact-CAD20250514licq_BSEP_DDA_60min_21pcw_3_PCP_T_AC_Slot2-42_1_7070.pep.xml
4 interact-CAD20250514licq_BSEP_DDA_60min_21pcw_3_PCP_T_AC_Slot2-42_1_7070.pep.xml
5 interact-CAD20250514licq_BSEP_DDA_60min_21pcw_3_PCP_T_AC_Slot2-42_1_7070.pep.xml
6 interact-CAD20250514licq_BSEP_DDA_60min_21pcw_3_PCP_T_AC_Slot2-42_1_7070.pep.xml
  Peptide                     Modified.Peptide           
1 YDGSTIVPGEQGAEYQHFIQQCTDDVR YDGSTIVPGEQGAEYQHFIQQCTDDVR
2 YDGSTIVPGEQGAEYQHFIQQCTDDVR YDGSTIVPGEQGAEYQHFIQQCTDDVR
3 YDGSTIVPGEQGAEYQHFIQQCTDDVR YDGSTIVPGEQGAEYQHFIQQCTDDVR
4 DREGFFTNGLTLGAK                                        
5 YDGSTIVPGEQGAEYQHFIQQCTDDVR YDGSTIVPGEQGAEYQHFIQQCTDDVR
6 EIQTAVR                                                
  Extended.Peptide                              Prev.AA Next.AA Peptide.Length
1 AVIWVTFK.YDGSTIVPGEQGAEYQHFIQQCTDDVR.LFAFVRFT K       L       27            
2 AVIWVTFK.YDGSTIVPGEQGAEYQHFIQQCTDDVR.LFAFVRFT K       L       27            
3 AVIWVTFK.YDGSTIVPGEQGAEYQHFIQQCTDDVR.LFAFVRFT K       L       27            
4 EIDMIVGK.DREGFFTNGLTLGAK.KCSVIRDS             K       K       15            
5 AVIWVTFK.YDGSTIVPGEQGAEYQHFIQQCTDDVR.LFAFVRFT K       L       27            
6 KRSTITSR.EIQTAVR.LIIPGEIA                     R       L        7            
  Charge Retention ⋯ Protein               Protein.ID Entry.Name  Gene  
1 3      2209.0874 ⋯ sp|Q14019|COTL1_HUMAN Q14019     COTL1_HUMAN COTL1 
2 2      2210.2747 ⋯ sp|Q14019|COTL1_HUMAN Q14019     COTL1_HUMAN COTL1 
3 2      2211.4846 ⋯ sp|Q14019|COTL1_HUMAN Q14019     COTL1_HUMAN COTL1 
4 3      2216.3074 ⋯ sp|P35080|PROF2_HUMAN P35080     PROF2_HUMAN PFN2  
5 3      2219.8826 ⋯ sp|Q14019|COTL1_HUMAN Q14019     COTL1_HUMAN COTL1 
6 1       554.2949 ⋯ sp|O60814|H2B1K_HUMAN O60814     H2B1K_HUMAN H2BC12
  Protein.Description   
1 Coactosin-like protein
2 Coactosin-like protein
3 Coactosin-like protein
4 Profilin-2            
5 Coactosin-like protein
6 Histone H2B type 1-K  
  Mapped.Genes                                                                                      
1                                                                                                   
2                                                                                                   
3                                                                                                   
4                                                                                                   
5                                                                                                   
6 H2BC1, H2BC11, H2BC12L, H2BC13, H2BC14, H2BC15, H2BC17, H2BC18, H2BC21, H2BC3, H2BC4, H2BC5, H2BC9
  Mapped.Proteins                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

# 非经典小肽

In [7]:
fread_c(total_psm_path) -> psm
# 非特异性
psm %>% filter(grepl("ENS",Protein)) %>% filter(Mapped.Proteins!="") %>% 
  filter(!grepl("sp",Mapped.Proteins)) -> psm_non_unique
psm_non_unique -> df
unfold(df) -> df_unfold
# 特异性（上个步骤已经导出）
fread_c(noncano_sep_psm_path) -> df_unique
df_unique$Group_id <- ""
df_unique$PSM_type <- "Unique"
df_unfold$PSM_type <- "NonUnique"
rbind(df_unique,df_unfold) -> psm_uncano_sep_all
psm_uncano_sep_all %>% group_by(PSM_type) %>% summarise(n_distinct=n_distinct(Protein))
fwrite(psm_uncano_sep_all,o("psm_uncano_sep_all.txt"),sep='\t')

PSM_type,n_distinct
<chr>,<int>
NonUnique,27157
Unique,2141


In [8]:
head(psm_uncano_sep_all)

,Spectrum,Spectrum.File,Peptide,Modified.Peptide,Extended.Peptide,Prev.AA,Next.AA,Peptide.Length,Charge,Retention,⋯,Protein,Protein.ID,Entry.Name,Gene,Protein.Description,Mapped.Genes,Mapped.Proteins,Sample,Group_id,PSM_type
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,CAD20241224licq_BSEP_DDA_60min_21pcw_1_less3K_Slot2-25_1_4706.01036.01036.2,interact-CAD20241224licq_BSEP_DDA_60min_21pcw_1_less3K_Slot2-25_1_4706.pep.xml,LRSASPTRSS,,.LRSASPTRSS.,-,-,10,2,414.4234,⋯,ENST00000457855.2+chr3:9739471-9739504,ENST00000457855.2+chr3:9739471-9739504 PE=4,ENST00000457855.2+chr3:9739471-9739504 PE=4,NA,ENST00000457855.2+chr3:9739471-9739504 PE=4,NA,NA,21pcw_1_less3K,,Unique
2,CAD20241224licq_BSEP_DDA_60min_21pcw_1_less3K_Slot2-25_1_4706.06453.06453.2,interact-CAD20241224licq_BSEP_DDA_60min_21pcw_1_less3K_Slot2-25_1_4706.pep.xml,APPGGRSNITSLS,,.APPGGRSNITSLS.,M,-,13,2,932.1971,⋯,ENST00000368627.1+chr10:132204885-132204930,ENST00000368627.1+chr10:132204885-132204930 PE=4,ENST00000368627.1+chr10:132204885-132204930 PE=4,NA,ENST00000368627.1+chr10:132204885-132204930 PE=4,NA,NA,21pcw_1_less3K,,Unique
3,CAD20241224licq_BSEP_DDA_60min_21pcw_1_less3K_Slot2-25_1_4706.06544.06544.2,interact-CAD20241224licq_BSEP_DDA_60min_21pcw_1_less3K_Slot2-25_1_4706.pep.xml,MKLEASQQNSEEM,,.MKLEASQQNSEEM.,-,-,13,2,943.9016,⋯,ENST00000538010.5+chr12:122059121-122059163,ENST00000538010.5+chr12:122059121-122059163 PE=4,ENST00000538010.5+chr12:122059121-122059163 PE=4,NA,ENST00000538010.5+chr12:122059121-122059163 PE=4,NA,NA,21pcw_1_less3K,,Unique
4,CAD20241224licq_BSEP_DDA_60min_21pcw_1_less3K_Slot2-25_1_4706.15833.15833.2,interact-CAD20241224licq_BSEP_DDA_60min_21pcw_1_less3K_Slot2-25_1_4706.pep.xml,LGGSAVISLEGKPL,,.LGGSAVISLEGKPL.,-,-,14,2,2245.5833,⋯,ENST00000534769.5-chr11:65855335-65855380,ENST00000534769.5-chr11:65855335-65855380 PE=4,ENST00000534769.5-chr11:65855335-65855380 PE=4,NA,ENST00000534769.5-chr11:65855335-65855380 PE=4,NA,NA,21pcw_1_less3K,,Unique
5,CAD20241224licq_BSEP_DDA_60min_21pcw_1_less3K_Slot2-25_1_4706.15955.15955.2,interact-CAD20241224licq_BSEP_DDA_60min_21pcw_1_less3K_Slot2-25_1_4706.pep.xml,LGGSAVISLEGKPL,,.LGGSAVISLEGKPL.,-,-,14,2,2252.6313,⋯,ENST00000534769.5-chr11:65855335-65855380,ENST00000534769.5-chr11:65855335-65855380 PE=4,ENST00000534769.5-chr11:65855335-65855380 PE=4,NA,ENST00000534769.5-chr11:65855335-65855380 PE=4,NA,NA,21pcw_1_less3K,,Unique
6,CAD20241224licq_BSEP_DDA_60min_21pcw_1_less3K_Slot2-25_1_4706.24850.24850.3,interact-CAD20241224licq_BSEP_DDA_60min_21pcw_1_less3K_Slot2-25_1_4706.pep.xml,TGEMSGPVFTDSGIHIILRTE,,.TGEMSGPVFTDSGIHIILRTE.,-,-,21,3,2714.5837,⋯,ENST00000586025.5+chr19:9849133-9849199,ENST00000586025.5+chr19:9849133-9849199 PE=4,ENST00000586025.5+chr19:9849133-9849199 PE=4,NA,ENST00000586025.5+chr19:9849133-9849199 PE=4,NA,NA,21pcw_1_less3K,,Unique


# 合并

In [9]:
rbind(df_unfold_target_add_PSM_type,psm_uncano_sep_all) -> psm_sep_all
fwrite(psm_sep_all,o("psm_sep_all.txt"),sep='\t')